In [12]:
import os
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta, timezone
import time

BASE_URL = 'https://api.nasa.gov/neo/rest/v1/feed'
API_KEY  = os.environ.get('NASA_API_KEY', 'DEMO_KEY')

now   = datetime.now(timezone.utc)
start = (now - timedelta(days=365)).strftime('%Y-%m-%d')
end   = (now + timedelta(days=180)).strftime('%Y-%m-%d')

print(f"Using API key: {'custom key set' if API_KEY != 'DEMO_KEY' else 'DEMO_KEY (rate limited)'}")
print(f"Fetching close approaches from {start} to {end}")

Using API key: DEMO_KEY (rate limited)
Fetching close approaches from 2025-03-23 to 2026-09-19


In [ ]:
now   = datetime.now(timezone.utc)
start = (now - timedelta(days=365)).strftime('%Y-%m-%d')
end   = (now + timedelta(days=180)).strftime('%Y-%m-%d')

API_KEY = os.environ.get('NASA_API_KEY', 'DEMO_KEY')

print("Fetching data... this will take a couple of minutes.")
df_neo = fetch_neows(start, end, api_key=API_KEY)
print(f"\nDone. {len(df_neo):,} close approaches fetched.")
print(df_neo.head())

Fetching data... this will take a couple of minutes.


KeyboardInterrupt: 

In [ ]:
# Clean and enrich
df_neo['date'] = pd.to_datetime(df_neo['date'])
df_neo['diameter_avg_m'] = (df_neo['diameter_min_m'] + df_neo['diameter_max_m']) / 2
df_neo['diameter_avg_km'] = df_neo['diameter_avg_m'] / 1000
df_neo['miss_lunar'] = df_neo['miss_lunar'].round(2)
df_neo['is_future'] = df_neo['date'] > datetime.now(timezone.utc).replace(tzinfo=None)

# Size category for colour coding
def size_category(d):
    if d < 50:    return 'Small (< 50m)'
    elif d < 200: return 'Medium (50-200m)'
    elif d < 500: return 'Large (200-500m)'
    else:         return 'Very Large (> 500m)'

df_neo['size_cat'] = df_neo['diameter_avg_m'].apply(size_category)

# Summary
print(f"Total close approaches: {len(df_neo):,}")
print(f"Potentially hazardous:  {df_neo['is_hazardous'].sum():,}")
print(f"Sentry objects:         {df_neo['is_sentry'].sum():,}")
print(f"Future approaches:      {df_neo['is_future'].sum():,}")
print(f"Past approaches:        {(~df_neo['is_future']).sum():,}")
print(f"\nSize breakdown:\n{df_neo['size_cat'].value_counts()}")
print(f"\nClosest approach: {df_neo['miss_km'].min():,.0f} km ({df_neo.loc[df_neo['miss_km'].idxmin(), 'name']})")
print(f"Fastest approach: {df_neo['velocity_km_s'].max():.1f} km/s ({df_neo.loc[df_neo['velocity_km_s'].idxmax(), 'name']})")

In [ ]:
size_colors = {
    'Small (< 50m)':        '#4fa3e0',
    'Medium (50-200m)':     '#fffcea',
    'Large (200-500m)':     '#ffd2a1',
    'Very Large (> 500m)':  '#ff6b6b'
}

fig_scatter = go.Figure()

for cat, color in size_colors.items():
    mask = df_neo['size_cat'] == cat
    subset = df_neo[mask]
    fig_scatter.add_trace(go.Scatter(
        x=subset['date'],
        y=subset['miss_lunar'],
        mode='markers',
        name=cat,
        marker=dict(
            color=color,
            size=np.clip(subset['diameter_avg_m'] / 20, 3, 20),
            opacity=0.7,
            line=dict(
                color=np.where(subset['is_hazardous'], 'red', 'rgba(0,0,0,0)'),
                width=np.where(subset['is_hazardous'], 1.5, 0)
            )
        ),
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Date: %{x|%d %b %Y}<br>'
            'Miss Distance: %{y:.1f} lunar distances<br>'
            'Diameter: %{customdata[1]:.0f}m (avg)<br>'
            'Velocity: %{customdata[2]:.1f} km/s<br>'
            'Hazardous: %{customdata[3]}'
            '<extra></extra>'
        ),
        customdata=list(zip(
            subset['name'],
            subset['diameter_avg_m'],
            subset['velocity_km_s'],
            subset['is_hazardous']
        ))
    ))

fig_scatter.add_hline(
    y=1, line_dash='dash', line_color='rgba(255,255,255,0.3)',
    annotation_text='Moon (1 LD)',
    annotation_font_color='rgba(255,255,255,0.5)'
)

fig_scatter.add_vline(
    x=datetime.now().timestamp() * 1000,
    line_dash='dash', line_color='rgba(255,255,255,0.3)',
    annotation_text='Today',
    annotation_font_color='rgba(255,255,255,0.5)'
)

fig_scatter.update_layout(
    title=dict(
        text='Near-Earth Asteroid Close Approaches  |  7,940 Objects',
        font=dict(size=20, color='white'), x=0.5
    ),
    paper_bgcolor='#04040f',
    plot_bgcolor='#04040f',
    xaxis=dict(
        title='Date', color='#888',
        gridcolor='#111128', zerolinecolor='#222'
    ),
    yaxis=dict(
        title='Miss Distance (Lunar Distances)',
        color='#888', gridcolor='#111128', zerolinecolor='#222',
        range=[0, 75]
    ),
    legend=dict(font=dict(color='white'), bgcolor='rgba(0,0,0,0.5)'),
    height=700,
    margin=dict(l=60, r=20, t=80, b=60)
)

fig_scatter.show()

In [ ]:
# Sort by date for animation
df_anim = df_neo[df_neo['miss_lunar'] <= 75].sort_values('date').reset_index(drop=True)

# Build one frame per month
df_anim['month'] = df_anim['date'].dt.to_period('M').astype(str)
months = sorted(df_anim['month'].unique())

frames = []
for i, month in enumerate(months):
    # Show all approaches up to and including this month
    visible = df_anim[df_anim['month'] <= month]
    current = df_anim[df_anim['month'] == month]

    frame_data = []

    # Historical points (faded)
    frame_data.append(go.Scatter(
        x=visible['date'],
        y=visible['miss_lunar'],
        mode='markers',
        marker=dict(
            color=visible['is_hazardous'].map({True: '#ff4444', False: '#4fa3e0'}),
            size=np.clip(visible['diameter_avg_m'] / 25, 3, 18),
            opacity=0.3,
        ),
        hoverinfo='skip',
        showlegend=False
    ))

    # Current month highlighted
    frame_data.append(go.Scatter(
        x=current['date'],
        y=current['miss_lunar'],
        mode='markers',
        marker=dict(
            color=current['is_hazardous'].map({True: '#ff4444', False: '#4fa3e0'}),
            size=np.clip(current['diameter_avg_m'] / 25, 4, 20),
            opacity=1.0,
            line=dict(color='white', width=1)
        ),
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Miss Distance: %{y:.1f} LD<br>'
            'Diameter: %{customdata[1]:.0f}m<br>'
            'Velocity: %{customdata[2]:.1f} km/s<br>'
            'Hazardous: %{customdata[3]}'
            '<extra></extra>'
        ),
        customdata=list(zip(
            current['name'],
            current['diameter_avg_m'],
            current['velocity_km_s'],
            current['is_hazardous']
        )),
        showlegend=False
    ))

    frames.append(go.Frame(
        data=frame_data,
        name=month,
        layout=go.Layout(title_text=f'Near-Earth Asteroid Close Approaches  |  {month}')
    ))

# Initial frame
fig_anim = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(
            text=f'Near-Earth Asteroid Close Approaches  |  {months[0]}',
            font=dict(size=18, color='white'), x=0.5
        ),
        paper_bgcolor='#04040f',
        plot_bgcolor='#04040f',
        xaxis=dict(
            title='Date', color='#888',
            gridcolor='#111128', zerolinecolor='#222',
            range=[df_anim['date'].min(), df_anim['date'].max()]
        ),
        yaxis=dict(
            title='Miss Distance (Lunar Distances)',
            color='#888', gridcolor='#111128', zerolinecolor='#222',
            range=[0, 75]
        ),
        height=700,
        margin=dict(l=60, r=20, t=80, b=60),
        updatemenus=[dict(
            type='buttons', showactive=False,
            y=1.08, x=0.5, xanchor='center',
            buttons=[
                dict(label='Play', method='animate',
                     args=[None, dict(frame=dict(duration=300, redraw=True),
                                      fromcurrent=True, transition=dict(duration=0))]),
                dict(label='Pause', method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate', transition=dict(duration=0))])
            ]
        )],
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix='Month: ', font=dict(color='white')),
            pad=dict(t=50),
            font=dict(color='white'),
            bgcolor='#1a1a2e',
            bordercolor='#333',
            steps=[dict(
                args=[[f.name], dict(frame=dict(duration=0, redraw=True), mode='immediate')],
                label=f.name, method='animate'
            ) for f in frames]
        )]
    )
)

fig_anim.add_hline(
    y=1, line_dash='dash', line_color='rgba(255,255,255,0.3)',
    annotation_text='Moon (1 LD)',
    annotation_font_color='rgba(255,255,255,0.5)'
)

fig_anim.show()

In [ ]:
fig_scatter.write_html('asteroid_approaches.html')
fig_anim.write_html('asteroid_animated.html')
print('Exported.')